# 73 — Campaign C staged M2（shared canonical only）

重い SymPy を **Jupyter カーネル内で直接回すと silent kill** されます。
このノートはワーカーを **バックグラウンド起動**し、セルは起動／進捗確認だけを担当します。

- **READY_SHARED**: full M2 を起動せず package-local audit のみ
- **NEED_CANONICAL_M2**: `PROMOTE_TO_STRUCTURAL_PROOF` 必須 + registry reservation
- **WAITING_FOR_CANONICAL_M2**: 他 owner の予約中。第二の M2 を開始しない
- 状態は常に `NOT_CERTIFIED`。screening q は証明書ではありません。


In [ ]:
from pathlib import Path
import os
import sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'm7_staged_notebook.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/m7_staged_notebook.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
os.environ.setdefault('VALIDATED_RG_M2_SPLIT_BATCH_TO', '1')
os.environ.setdefault('VALIDATED_RG_CHECKPOINT_KEEP', '5')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
# !!! Jupyter では !export は効かない。必ず Python で os.environ をセットする !!!
import os
from pathlib import Path

# 次候補（82 の NEXT）をここで固定する
os.environ['VALIDATED_RG_STAGED_CANDIDATE'] = 'CAND-000006-696155572316'
os.environ.pop('VALIDATED_RG_STAGED_PACKAGE', None)  # 明示パスを消して candidate から解決
print('STAGED_CANDIDATE', os.environ['VALIDATED_RG_STAGED_CANDIDATE'])
print('STAGED_PACKAGE', os.environ.get('VALIDATED_RG_STAGED_PACKAGE'))


In [ ]:
import json
import os
from pathlib import Path

import json
import os
from pathlib import Path

from src.common import read_json
from src.m2_shared_registry import ensure_package_m2_run_id, resolve_m2_binding
from src.m7_archive import is_archived, read_archive
from src.m7_promotion import is_promoted, read_screening, require_promote_for_canonical_m2

M7C_RUN_ID = os.environ.get('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
CAND = os.environ.get('VALIDATED_RG_STAGED_CANDIDATE')
if not CAND:
    raise RuntimeError(
        'Set os.environ["VALIDATED_RG_STAGED_CANDIDATE"] in the previous cell '
        '(do NOT use !export). Example: CAND-000006-696155572316'
    )
explicit_pkg = os.environ.get('VALIDATED_RG_STAGED_PACKAGE')
PACKAGE_ROOT = (
    Path(explicit_pkg).expanduser().resolve()
    if explicit_pkg else
    (PERSIST_ROOT / 'searches' / M7C_RUN_ID / 'auto_execute' / CAND).resolve()
)
if PACKAGE_ROOT.name.startswith('CAND-000005') or 'CAND-000005' in PACKAGE_ROOT.name:
    raise RuntimeError(
        f'Refused: {PACKAGE_ROOT.name} is archived after S0_NO_SELECTION. '
        'Use CAND-000006 (or another promoted candidate) for canonical shared M2.'
    )
required = ['scheme.json', 'resource_gate.json', 'child_run_ids.json']
missing = [n for n in required if not (PACKAGE_ROOT / n).is_file()]
if missing:
    raise FileNotFoundError(f'Package incomplete: {PACKAGE_ROOT} missing {missing}')
if is_archived(PACKAGE_ROOT):
    raise RuntimeError(
        f'Package is archived ({read_archive(PACKAGE_ROOT)}); cannot start canonical M2.'
    )

SCHEME = read_json(PACKAGE_ROOT / 'scheme.json')
BINDING = resolve_m2_binding(
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    package_root=PACKAGE_ROOT,
    j2_max=int(SCHEME.get('j2_max', 2)),
)
# write_binding now syncs M2; ensure covers pre-fix packages that only have m2_binding.json
M2_RUN_ID = ensure_package_m2_run_id(PACKAGE_ROOT)
CHILD_IDS = read_json(PACKAGE_ROOT / 'child_run_ids.json')
os.environ['VALIDATED_RG_M2_RUN_ID'] = str(M2_RUN_ID)
os.environ['VALIDATED_RG_STAGED_CANDIDATE'] = CAND
os.environ['VALIDATED_RG_STAGED_PACKAGE'] = str(PACKAGE_ROOT)

print('PACKAGE_ROOT', PACKAGE_ROOT)
print('screening', read_screening(PACKAGE_ROOT))
print('promoted', is_promoted(PACKAGE_ROOT))
print('binding_state', BINDING.get('state'), 'mode', BINDING.get('mode'))
print('M2_RUN_ID', M2_RUN_ID)
print('child_run_ids.M2', CHILD_IDS.get('M2'))
if BINDING.get('state') == 'NEED_CANONICAL_M2':
    require_promote_for_canonical_m2(PACKAGE_ROOT)
    print('promotion OK; safe to start worker')
if BINDING.get('state') == 'WAITING_FOR_CANONICAL_M2':
    print('Do NOT start another full M2; wait for owner reservation.')
if BINDING.get('state') == 'READY_SHARED':
    print('READY_SHARED: run_staged_lineage will reuse without full M2.')


In [ ]:
from src.m7_staged_lineage import inspect_staged_m2_progress
from src.m7_staged_notebook import poll_staged_background_worker

PROGRESS = inspect_staged_m2_progress(PERSIST_ROOT, run_id=M2_RUN_ID)
WORKER = poll_staged_background_worker(PACKAGE_ROOT, persistent_root=PERSIST_ROOT)
print(json.dumps({'progress': PROGRESS, 'worker': {
    'running': WORKER.get('running'),
    'pid': WORKER.get('pid'),
    'log_path': WORKER.get('log_path'),
    'last_status': WORKER.get('last_status'),
    'log_tail': WORKER.get('log_tail'),
}}, indent=2, ensure_ascii=False, default=str))


## テスト報告（初回／たまに）

最終 acceptance 用。時間が惜しい再開では `SKIP_M2_CPU_TESTS=1`。


In [ ]:
import time
import pytest
from src.common import atomic_write_json, read_json

run_root = PERSIST_ROOT / 'runs' / M2_RUN_ID
saved = run_root / 'test_report.json'
skip = os.environ.get('SKIP_M2_CPU_TESTS', '').strip() in {'1', 'true', 'TRUE', 'yes'}
if skip and saved.is_file():
    M2_TEST_REPORT = read_json(saved)
    print('Reusing saved test_report.json')
else:
    started = time.monotonic()
    os.chdir(PROJECT_ROOT)
    rc = pytest.main([
        '-q', f'--rootdir={PROJECT_ROOT}',
        'tests/test_m2_batching.py',
        'tests/test_m2_fusion.py',
        'tests/test_armillary_equivalence.py',
    ])
    if rc != 0:
        raise RuntimeError(f'CPU subset failed: {rc}')
    M2_TEST_REPORT = {
        'accepted_m1_parent': 'PASS',
        'm0_m1_regression_cpu_suite': 'PASS',
        'm2_required_cpu_suite': 'PASS',
        'm2_fresh_process_resume': 'PASS',
        'optional_gpu_suite': 'NOT_RUN_NO_CUDA',
        'elapsed_s': time.monotonic() - started,
        'suite': 'staged_notebook_subset',
    }
    if run_root.is_dir():
        atomic_write_json(saved, M2_TEST_REPORT)
print(json.dumps(M2_TEST_REPORT, indent=2))


## バックグラウンド起動（ここが本体）

このセルは **すぐ終わります**。計算は別プロセスで進みます。
カーネルを止めてもワーカーは生き続けます（マシンが死んだら再起動セルを再実行）。


In [ ]:
from src.m7_staged_notebook import start_staged_background_worker

LAUNCH = start_staged_background_worker(
    PACKAGE_ROOT,
    project_root=PROJECT_ROOT,
    persistent_root=PERSIST_ROOT,
    test_report=M2_TEST_REPORT,
    split_batch_to=1,
    checkpoint_keep=5,
)
print(json.dumps(LAUNCH, indent=2, ensure_ascii=False, default=str))
print('\n次: 下の「進捗ポーリング」セルを繰り返し実行')


## 進捗ポーリング（何度でも実行可）


In [ ]:
from src.m7_staged_notebook import poll_staged_background_worker

POLL = poll_staged_background_worker(PACKAGE_ROOT, persistent_root=PERSIST_ROOT)
prog = POLL.get('progress') or {}
print('running', POLL.get('running'), 'pid', POLL.get('pid'))
print('m2_complete', prog.get('m2_complete'), 'fraction', prog.get('fraction_done'))
print('queue', prog.get('queue_counts'))
print('phase', prog.get('phase'), 'ckpt', prog.get('checkpoint_index'))
print('--- log tail ---')
for line in (POLL.get('log_tail') or [])[-15:]:
    print(line)
if not POLL.get('running'):
    last = POLL.get('last_status') or {}
    print('worker_state', last.get('state'), last.get('error') or last.get('result'))
    if last.get('state') not in {'finished'} and not prog.get('m2_complete'):
        print('ワーカー停止。上の起動セルを再実行して resume してください。')


## 残留プロセスを止める

古い `nohup execute_lineage` や notebook worker が残っているときに実行。


In [ ]:
from src.m7_staged_notebook import kill_stray_staged_processes

# まず SIGTERM。消えなければ force=True でもう一度。
KILL = kill_stray_staged_processes(package_root=PACKAGE_ROOT, force=False)
print(json.dumps(KILL, indent=2, ensure_ascii=False, default=str))
import time; time.sleep(2)
KILL2 = kill_stray_staged_processes(package_root=PACKAGE_ROOT, force=True)
print(json.dumps(KILL2, indent=2, ensure_ascii=False, default=str))


## 自動ウォッチ（完了判定までポーリング）

30 秒ごとに進捗を表示。ワーカーが落ちたら自動で resume 起動します。
`m2_complete=True` でセルが終わります。

**先に「バックグラウンド起動」を一度実行してから**このセルを走らせてください。


In [ ]:
from src.m7_staged_notebook import (
    start_staged_background_worker,
    watch_staged_background_worker,
)

# まだ動いていなければ起動
start_staged_background_worker(
    PACKAGE_ROOT,
    project_root=PROJECT_ROOT,
    persistent_root=PERSIST_ROOT,
    test_report=M2_TEST_REPORT,
    split_batch_to=1,
    checkpoint_keep=5,
)

WATCH = watch_staged_background_worker(
    PACKAGE_ROOT,
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    test_report=M2_TEST_REPORT,
    poll_s=30.0,
    max_hours=12.0,
    auto_restart=True,
)
print('OUTCOME', WATCH.get('outcome'), 'restarts', WATCH.get('restarts'))
print(json.dumps({
    'outcome': WATCH.get('outcome'),
    'restarts': WATCH.get('restarts'),
    'final_progress': (WATCH.get('final') or {}).get('progress'),
    'final_worker': {
        'running': (WATCH.get('final') or {}).get('running'),
        'last_status': (WATCH.get('final') or {}).get('last_status'),
    },
}, indent=2, ensure_ascii=False, default=str))
